EMBEDDING TYPES: Text data Embedding using qdrant vector database

In [23]:
from qdrant_client import QdrantClient

# Connect to a local Qdrant instance
client = QdrantClient("localhost", port=6333)


In [4]:
client.recreate_collection(
    collection_name="my_dense_vectors",
    vectors_config={"size": 3, "distance": "Cosine"}  # 3D vector, cosine similarity
)


C:\Users\admin\AppData\Local\Temp\ipykernel_13304\1800775200.py:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [6]:
from qdrant_client.models import PointStruct

# Insert vectors into the collection
client.upsert(
    collection_name="my_dense_vectors",
    points=[
        PointStruct(id=1, vector=[0.1, 0.2, 0.3], payload={"name": "vector1"}),
        PointStruct(id=2, vector=[0.4, 0.5, 0.6], payload={"name": "vector2"}),
        PointStruct(id=3, vector=[0.7, 0.8, 0.9], payload={"name": "vector3"}),
    ]
)


UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [8]:
# Search for the most similar vector
search_results = client.search(
    collection_name="my_dense_vectors",
    query_vector=[0.1, 0.2, 0.3],
    limit=2  # Get top 2 similar vectors
)

# Print search results
for result in search_results:
    print(f"ID: {result.id}, Score: {result.score}, Payload: {result.payload}")


ID: 1, Score: 0.9999998, Payload: {'name': 'vector1'}
ID: 2, Score: 0.9746317, Payload: {'name': 'vector2'}


C:\Users\admin\AppData\Local\Temp\ipykernel_13304\2519119336.py:2: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_results = client.search(


In [10]:
all_vectors = client.scroll(
    collection_name="my_dense_vectors",
    limit=10
)

print(all_vectors)


([Record(id=1, payload={'name': 'vector1'}, vector=None, shard_key=None, order_value=None), Record(id=2, payload={'name': 'vector2'}, vector=None, shard_key=None, order_value=None), Record(id=3, payload={'name': 'vector3'}, vector=None, shard_key=None, order_value=None)], None)


In [12]:
from sentence_transformers import SentenceTransformer

# Load a pre-trained Sentence Transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')  # Lightweight and efficient


In [13]:
sentences = [
    "Artificial Intelligence is transforming the world.",
    "Machine learning models improve with more data.",
    "Quantum computing is the future of technology."
]

# Generate embeddings (vectors)
embeddings = model.encode(sentences, convert_to_numpy=True)

# Print vector for the first sentence
print(embeddings[0])  # A 384-dimensional vector


[ 3.87241468e-02 -1.10552402e-03  8.27161819e-02 -1.62886064e-02
  4.65431400e-02 -9.53026768e-03 -2.99749039e-02  3.49415140e-03
  1.11962436e-02  2.63022631e-03 -1.33261634e-02  7.20948651e-02
 -4.09049820e-03  4.32347283e-02 -3.58305424e-02  3.18855755e-02
 -1.07398152e-01 -3.44287939e-02 -1.47344694e-01 -8.24283734e-02
  9.44172964e-03  4.44823727e-02 -1.04593318e-02 -2.27152240e-02
 -6.94131339e-03  5.52090257e-02  2.38689650e-02 -6.28514439e-02
 -2.65044570e-02 -5.03157675e-02  1.60141867e-02  6.18262738e-02
  3.32975611e-02  2.48469673e-02 -2.86043976e-02  8.12884644e-02
  2.73210779e-02 -1.45991426e-02  6.13124780e-02 -4.10649851e-02
  3.10627576e-02 -8.35286304e-02  6.51204586e-03 -2.13338882e-02
  6.93308040e-02  1.36500552e-01 -7.67961293e-02 -3.25734131e-02
 -1.00621663e-03 -3.51937674e-03 -1.30318135e-01 -3.58479507e-02
  2.11035572e-02 -1.34975417e-02 -3.76569442e-02  2.81999689e-02
  9.11419392e-02 -1.88492592e-02  5.50939888e-02 -3.09680328e-02
  1.53997242e-02 -1.05273

In [14]:
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct

# Connect to Qdrant (running locally)
client = QdrantClient("localhost", port=6333)

# Create a collection to store 384-dimensional vectors
client.recreate_collection(
    collection_name="text_embeddings",
    vectors_config={"size": 384, "distance": "Cosine"}
)


C:\Users\admin\AppData\Local\Temp\ipykernel_13304\4251877872.py:8: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [18]:
# Insert vectors into Qdrant
client.upsert(
    collection_name="text_embeddings",
    points=[
        PointStruct(id=i, vector=embeddings[i], payload={"text": sentences[i]})
        for i in range(len(sentences))
    ]
)


UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [20]:
# Query a new sentence
query_text = "AI is changing the world."
query_vector = model.encode([query_text])[0]

# Search for the most similar stored text
search_results = client.search(
    collection_name="text_embeddings",
    query_vector=query_vector,
    limit=2
)

# Print results
for result in search_results:
    print(f"Text: {result.payload['text']}, Score: {result.score}")


Text: Artificial Intelligence is transforming the world., Score: 0.83032155
Text: Machine learning models improve with more data., Score: 0.43522522


C:\Users\admin\AppData\Local\Temp\ipykernel_13304\1192363336.py:6: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_results = client.search(
